# Gap-Level Policy Compliance Detection using Multilingual BERT

**Multi-label classification**: Given a policy excerpt, detect which of 16 compliance gaps are present.

This replaces the old approach of classifying an entire document with a single label. Real policy documents contain multiple domains (password policy, risk assessment, etc.) and can have multiple gaps simultaneously.

**Model**: `bert-base-multilingual-cased` → 16 sigmoid outputs (one per gap)
**Loss**: `BCEWithLogitsLoss` (binary cross-entropy per gap label)
**Standards**: NCA ECC-2:2024 + ISO 27001:2022

| Gap ID | Domain | Description |
|--------|--------|-------------|
| GAP_PP_001-008 | Password Policy (ECC 2-2) | Complexity, expiration, lockout, MFA, PAM, encryption, review, roles |
| GAP_RA_001-008 | Risk Assessment (ECC 1-5) | Methodology, identification, scales, treatment, triggers, register, review, integration |

**Inference Pipeline**: Document → Chunk → Per-chunk gap detection → Aggregate → Report

In [ ]:
# Install required packages
!pip install transformers datasets torch scikit-learn accelerate -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

d:\iso-policy-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 1. Load Dataset & Define Constants

In [ ]:
# ========== GAP LABELS (16 gaps) ==========
GAP_LABELS = [
    "GAP_PP_001", "GAP_PP_002", "GAP_PP_003", "GAP_PP_004",
    "GAP_PP_005", "GAP_PP_006", "GAP_PP_007", "GAP_PP_008",
    "GAP_RA_001", "GAP_RA_002", "GAP_RA_003", "GAP_RA_004",
    "GAP_RA_005", "GAP_RA_006", "GAP_RA_007", "GAP_RA_008",
]
GAP_TO_IDX = {g: i for i, g in enumerate(GAP_LABELS)}
NUM_GAPS = len(GAP_LABELS)

GAP_DESCRIPTIONS = {
    "GAP_PP_001": "Weak password complexity requirements",
    "GAP_PP_002": "Inadequate password expiration policy",
    "GAP_PP_003": "Weak account lockout policy",
    "GAP_PP_004": "Missing Multi-Factor Authentication (MFA)",
    "GAP_PP_005": "Missing Privileged Access Management (PAM)",
    "GAP_PP_006": "Missing password encryption/storage requirements",
    "GAP_PP_007": "Missing periodic review schedule (password policy)",
    "GAP_PP_008": "Missing or vague roles and responsibilities (password policy)",
    "GAP_RA_001": "Missing documented risk methodology",
    "GAP_RA_002": "Missing risk identification procedures",
    "GAP_RA_003": "Missing likelihood/impact assessment scales",
    "GAP_RA_004": "Missing risk treatment options",
    "GAP_RA_005": "Missing mandatory risk assessment triggers",
    "GAP_RA_006": "Missing risk register requirements",
    "GAP_RA_007": "Missing periodic review schedule (risk assessment)",
    "GAP_RA_008": "Missing integration with project management",
}

# ========== LOAD DATASET ==========
dataset_path = "dataset/compliance_dataset.json"
with open(dataset_path, 'r', encoding='utf-8') as f:
    samples = json.load(f)

print(f"Loaded {len(samples)} samples")
print(f"Sample keys: {list(samples[0].keys())}")

# ========== DATASET STATISTICS ==========
# Compliance distribution
compliance_dist = {}
for s in samples:
    c = s["overall_compliance"]
    compliance_dist[c] = compliance_dist.get(c, 0) + 1
print(f"\nCompliance distribution: {compliance_dist}")

# Language distribution
lang_dist = {}
for s in samples:
    lang_dist[s["language"]] = lang_dist.get(s["language"], 0) + 1
print(f"Language distribution: {lang_dist}")

# Gap frequency
gap_vectors = np.array([
    [s["gap_labels"].get(g, 0) for g in GAP_LABELS] for s in samples
], dtype=np.float32)

print(f"\nGap frequencies (out of {len(samples)} samples):")
for i, g in enumerate(GAP_LABELS):
    count = int(gap_vectors[:, i].sum())
    bar = "█" * int(count / len(samples) * 30)
    print(f"  {g}: {count:>3} ({count/len(samples):>5.1%}) {bar}")

print(f"\nAvg gaps per sample: {gap_vectors.sum(axis=1).mean():.1f}")
print(f"Label density: {gap_vectors.mean():.2%}")
print(f"Samples with 0 gaps: {int((gap_vectors.sum(axis=1) == 0).sum())}")
print(f"Samples with 5+ gaps: {int((gap_vectors.sum(axis=1) >= 5).sum())}")

Loaded 577 samples

Sample keys: ['id', 'policy_type', 'language', 'quality_level', 'policy_text', 'label', 'compliance_score', 'gaps_found', 'gap_count', 'analysis_summary', 'ecc_control_reference', 'ecc_sub_controls', 'iso_controls', 'generated_at', 'model_used']

Label distribution:
  Fully Compliant: 202 (35.0%)
  Non-Compliant: 173 (30.0%)
  Partially Compliant: 202 (35.0%)

Language distribution:
  ar: 284
  en: 293

Quality level distribution:
  excellent: 202
  partial: 202
  weak: 173


## 2. Prepare Training Data

Each sample → 16-dim binary vector (multi-hot). Split into train/val/test with stratification on gap count.

In [ ]:
# Extract texts and labels
texts = [s["policy_excerpt"] for s in samples]
labels = gap_vectors  # shape: (N, 16)

# Stratify on gap count (binned) to keep distribution balanced across splits
gap_counts = labels.sum(axis=1).astype(int)
# Bin into categories for stratification: 0, 1-2, 3-5, 6+
strat_bins = np.where(gap_counts == 0, 0,
             np.where(gap_counts <= 2, 1,
             np.where(gap_counts <= 5, 2, 3)))

# Split: 70% train, 15% val, 15% test
indices = list(range(len(samples)))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, random_state=42, stratify=strat_bins
)
temp_strat = strat_bins[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=42, stratify=temp_strat
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
print(f"Train gap distribution: {np.bincount(strat_bins[train_idx], minlength=4)}")
print(f"Val gap distribution:   {np.bincount(strat_bins[val_idx], minlength=4)}")
print(f"Test gap distribution:  {np.bincount(strat_bins[test_idx], minlength=4)}")

Train samples: 403
Validation samples: 87
Test samples: 87


## 3. Tokenizer & DataLoaders

In [ ]:
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 512
BATCH_SIZE = 4

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)


class GapDetectionDataset(TorchDataset):
    """Dataset for multi-label gap detection."""

    def __init__(self, indices, all_texts, all_labels, tokenizer, max_length=512):
        self.texts = [all_texts[i] for i in indices]
        self.labels = [all_labels[i] for i in indices]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }


# Create datasets and loaders
train_dataset = GapDetectionDataset(train_idx, texts, labels, tokenizer, MAX_LENGTH)
val_dataset = GapDetectionDataset(val_idx, texts, labels, tokenizer, MAX_LENGTH)
test_dataset = GapDetectionDataset(test_idx, texts, labels, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Sanity check
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_ids:     {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  labels:        {batch['labels'].shape}")
print(f"  labels sample: {batch['labels'][0].tolist()}")

Map: 100%|██████████| 87/87 [00:00<00:00, 365.75 examples/s]

Tokenization complete!
Train dataset: Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 403
})
Validation dataset: Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 87
})


## 4. Model Architecture

mBERT backbone → single gap detection head with 16 sigmoid outputs.

```
Input (512 tokens) → BERT Encoder → [CLS] pooled (768) → Linear(768→256) → ReLU → Dropout → Linear(256→16) → Sigmoid
```

Each output is independent (not softmax) — a chunk can have 0 gaps, 3 gaps, or all 16.

In [ ]:
class GapDetectionModel(nn.Module):
    """
    Multi-label gap detection model.
    mBERT backbone + classification head with 16 sigmoid outputs.
    """

    def __init__(self, model_name="bert-base-multilingual-cased",
                 num_gaps=16, dropout_rate=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 768

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_gaps)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS] token, shape: (batch, 768)
        logits = self.classifier(pooled)  # shape: (batch, 16)
        return logits  # raw logits — apply sigmoid at inference


# Initialize
model = GapDetectionModel(
    model_name=MODEL_NAME,
    num_gaps=NUM_GAPS,
    dropout_rate=0.3
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Output: {NUM_GAPS} gap labels (multi-label, sigmoid + BCE)")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 427.08it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

Model loaded: bert-base-multilingual-cased
Number of parameters: 177,855,747


## 5. Training

BCEWithLogitsLoss (sigmoid built into the loss for numerical stability). Training with AdamW + linear warmup scheduler + gradient clipping.

In [ ]:
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5

loss_fn = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


def train_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, loss_fn, threshold=0.5):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        total_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).int()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.int().cpu().numpy())

    preds_arr = np.array(all_preds)
    labels_arr = np.array(all_labels)

    # Metrics
    hamming_acc = (preds_arr == labels_arr).mean()
    exact_match = (preds_arr == labels_arr).all(axis=1).mean()
    macro_f1 = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)
    micro_f1 = f1_score(labels_arr, preds_arr, average='micro', zero_division=0)

    return {
        "loss": total_loss / len(loader),
        "hamming_acc": hamming_acc,
        "exact_match": exact_match,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
    }


print(f"Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Loss: BCEWithLogitsLoss (multi-label)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training configuration ready!


## 6. Run Training Loop

In [ ]:
print("=" * 65)
print("TRAINING: Multi-Label Gap Detection")
print("=" * 65)

best_val_f1 = 0.0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn)
    val_metrics = evaluate(model, val_loader, loss_fn)

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"  Train Loss:    {train_loss:.4f}")
    print(f"  Val Loss:      {val_metrics['loss']:.4f}")
    print(f"  Val Macro F1:  {val_metrics['macro_f1']:.4f}")
    print(f"  Val Micro F1:  {val_metrics['micro_f1']:.4f}")
    print(f"  Val Hamming:   {val_metrics['hamming_acc']:.4f}")
    print(f"  Val Exact Match: {val_metrics['exact_match']:.4f}")

    if val_metrics['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['macro_f1']
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "./gap_model_best.pt")
        print(f"  >>> Best model saved! (Macro F1: {best_val_f1:.4f})")

print(f"\nBest model from epoch {best_epoch} (Macro F1: {best_val_f1:.4f})")
model.load_state_dict(torch.load("./gap_model_best.pt", map_location=device))
print("Best model loaded.")

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.528449,0.631845,0.793103


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.64s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

Training complete!


## 7. Evaluate on Test Set

Per-gap Precision/Recall/F1, plus aggregate metrics (Macro F1, Micro F1, Exact Match, Hamming Accuracy).

In [ ]:
# Collect all test predictions
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].int().numpy())

preds_arr = np.array(all_preds)
labels_arr = np.array(all_labels)
probs_arr = np.array(all_probs)

# ========== PER-GAP METRICS ==========
print("=" * 65)
print("PER-GAP METRICS (Test Set)")
print("=" * 65)
print(f"\n{'Gap ID':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 55)

for i, gap in enumerate(GAP_LABELS):
    true_col = labels_arr[:, i]
    pred_col = preds_arr[:, i]
    support = int(true_col.sum())
    if support > 0:
        p = precision_score(true_col, pred_col, zero_division=0)
        r = recall_score(true_col, pred_col, zero_division=0)
        f = f1_score(true_col, pred_col, zero_division=0)
        print(f"{gap:<15} {p:>10.3f} {r:>10.3f} {f:>10.3f} {support:>10}")
    else:
        print(f"{gap:<15} {'N/A':>10} {'N/A':>10} {'N/A':>10} {0:>10}")

# ========== AGGREGATE METRICS ==========
macro_f1 = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)
micro_f1 = f1_score(labels_arr, preds_arr, average='micro', zero_division=0)
exact_match = (preds_arr == labels_arr).all(axis=1).mean()
hamming = (preds_arr == labels_arr).mean()

print(f"\n{'='*55}")
print(f"Macro F1:       {macro_f1:.4f}")
print(f"Micro F1:       {micro_f1:.4f}")
print(f"Exact Match:    {exact_match:.4f}")
print(f"Hamming Acc:    {hamming:.4f}")

# ========== DOMAIN-LEVEL SUMMARY ==========
print(f"\n{'='*55}")
print("DOMAIN-LEVEL SUMMARY")
pp_f1 = f1_score(labels_arr[:, :8], preds_arr[:, :8], average='macro', zero_division=0)
ra_f1 = f1_score(labels_arr[:, 8:], preds_arr[:, 8:], average='macro', zero_division=0)
print(f"Password Policy (GAP_PP) Macro F1: {pp_f1:.4f}")
print(f"Risk Assessment (GAP_RA) Macro F1: {ra_f1:.4f}")

CLASSIFICATION REPORT
                     precision    recall  f1-score   support

    Fully Compliant       0.91      1.00      0.95        31
Partially Compliant       1.00      0.50      0.67        30
      Non-Compliant       0.68      1.00      0.81        26

           accuracy                           0.83        87
          macro avg       0.87      0.83      0.81        87
       weighted avg       0.87      0.83      0.81        87


Confusion Matrix:
[[31  0  0]
 [ 3 15 12]
 [ 0  0 26]]


## 8. Save Model

In [ ]:
import os

SAVE_PATH = "./policy_gap_detector"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), os.path.join(SAVE_PATH, "model.pt"))

# Save tokenizer
tokenizer.save_pretrained(SAVE_PATH)

# Save config (needed to reconstruct model at inference)
config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_gaps": NUM_GAPS,
    "gap_labels": GAP_LABELS,
    "gap_descriptions": GAP_DESCRIPTIONS,
    "dropout_rate": 0.3,
    "threshold": 0.5,
}
with open(os.path.join(SAVE_PATH, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"Model saved to: {SAVE_PATH}/")
for fname in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(os.path.join(SAVE_PATH, fname))
    print(f"  {fname}: {size / 1024:.1f} KB")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.42s/it]

Model saved to: ./policy_compliance_classifier


## 9. Full Document Inference Pipeline

Real documents are too long for BERT's 512-token limit. The pipeline:

1. **Chunk**: Split document into sections (~300-400 words each)
2. **Predict**: Run each chunk through the model → 16 gap probabilities per chunk
3. **Aggregate**: For each gap, take MAX probability across all chunks
4. **Report**: Generate domain-level compliance assessment with specific findings

In [ ]:
import re

# ========== DOCUMENT CHUNKING ==========

def chunk_document(text, max_chunk_words=350, overlap_words=50):
    """Split a full document into overlapping chunks by section headings."""
    # Try splitting on headings
    section_pattern = r'(?=\n#{1,4}\s|\n\d+[\.\)]\s|\n[١-٩][\.\-])'
    sections = re.split(section_pattern, text)
    sections = [s.strip() for s in sections if s.strip()]

    # Fallback to paragraphs
    if len(sections) <= 1:
        sections = re.split(r'\n\s*\n', text)
        sections = [s.strip() for s in sections if s.strip()]

    # Merge small sections, split large ones
    chunks = []
    current = ""
    for section in sections:
        if len(section.split()) > max_chunk_words:
            if current:
                chunks.append(current)
                current = ""
            # Split large section by word count
            words = section.split()
            start = 0
            while start < len(words):
                end = min(start + max_chunk_words, len(words))
                chunks.append(" ".join(words[start:end]))
                start += max_chunk_words - overlap_words
        else:
            combined = (current + "\n\n" + section).strip() if current else section
            if len(combined.split()) > max_chunk_words:
                if current:
                    chunks.append(current)
                current = section
            else:
                current = combined
    if current:
        chunks.append(current)

    # Merge tiny chunks (<20 words) into previous
    final = []
    for chunk in chunks:
        if len(chunk.split()) < 20 and final:
            final[-1] += "\n\n" + chunk
        else:
            final.append(chunk)

    return final if final else [text]


# ========== PER-CHUNK PREDICTION ==========

@torch.no_grad()
def predict_chunk(text, model, tokenizer, max_length=512):
    """Predict gap probabilities for a single chunk."""
    model.eval()
    inputs = tokenizer(
        text, padding="max_length", truncation=True,
        max_length=max_length, return_tensors="pt"
    ).to(device)
    logits = model(inputs["input_ids"], inputs["attention_mask"])
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    return {gap: float(probs[i]) for i, gap in enumerate(GAP_LABELS)}


# ========== AGGREGATION + REPORT ==========

def analyze_document(document_text, model, tokenizer, threshold=0.5):
    """
    Full document analysis pipeline.
    Returns structured compliance report with per-domain gap findings.
    """
    # 1. Chunk
    chunks = chunk_document(document_text)

    # 2. Predict per chunk
    chunk_results = []
    for chunk in chunks:
        probs = predict_chunk(chunk, model, tokenizer)
        chunk_results.append(probs)

    # 3. Aggregate: max probability across chunks for each gap
    aggregated = {}
    for gap in GAP_LABELS:
        max_prob = max(cr[gap] for cr in chunk_results)
        aggregated[gap] = {
            "probability": round(max_prob, 4),
            "detected": max_prob >= threshold,
        }

    # 4. Build report
    pp_gaps = [g for g in GAP_LABELS[:8] if aggregated[g]["detected"]]
    ra_gaps = [g for g in GAP_LABELS[8:] if aggregated[g]["detected"]]
    all_gaps = pp_gaps + ra_gaps

    # Derive compliance level
    if len(all_gaps) == 0:
        compliance = "compliant"
    elif len(all_gaps) >= 6:
        compliance = "non_compliant"
    else:
        compliance = "partially_compliant"

    # Score: 1.0 minus weighted penalty
    severity_weights = {
        "GAP_PP_001": 3, "GAP_PP_002": 2, "GAP_PP_003": 2, "GAP_PP_004": 4,
        "GAP_PP_005": 4, "GAP_PP_006": 3, "GAP_PP_007": 2, "GAP_PP_008": 2,
        "GAP_RA_001": 4, "GAP_RA_002": 3, "GAP_RA_003": 3, "GAP_RA_004": 3,
        "GAP_RA_005": 4, "GAP_RA_006": 2, "GAP_RA_007": 2, "GAP_RA_008": 2,
    }
    total_weight = sum(severity_weights.values())
    penalty = sum(severity_weights[g] for g in all_gaps)
    score = round(max(0.0, 1.0 - penalty / total_weight), 2)

    return {
        "overall_compliance": compliance,
        "overall_score": score,
        "gap_count": len(all_gaps),
        "num_chunks": len(chunks),
        "password_policy": {
            "gaps_detected": pp_gaps,
            "gap_count": len(pp_gaps),
            "score": round(1.0 - len(pp_gaps) / 8, 2),
            "details": [
                {"gap_id": g, "description": GAP_DESCRIPTIONS[g],
                 "confidence": aggregated[g]["probability"]}
                for g in pp_gaps
            ],
        },
        "risk_assessment": {
            "gaps_detected": ra_gaps,
            "gap_count": len(ra_gaps),
            "score": round(1.0 - len(ra_gaps) / 8, 2),
            "details": [
                {"gap_id": g, "description": GAP_DESCRIPTIONS[g],
                 "confidence": aggregated[g]["probability"]}
                for g in ra_gaps
            ],
        },
        "all_gap_probabilities": {g: aggregated[g]["probability"] for g in GAP_LABELS},
    }


print("Inference pipeline ready.")

Testing inference on a sample...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 578.02it/s, Materializing param=classifier.weight]                                      


Predicted: Non-Compliant
Confidence: 64.81%
All scores: {'Fully Compliant': 0.09050708264112473, 'Partially Compliant': 0.2614160478115082, 'Non-Compliant': 0.6480768322944641}


## 10. Demo: Test on Sample Documents

Run the full pipeline on test excerpts to verify end-to-end behavior.

In [ ]:
# Test on samples from the test set
print("=" * 65)
print("DEMO: Full Pipeline on Test Samples")
print("=" * 65)

for i in range(min(3, len(test_idx))):
    idx = test_idx[i]
    sample = samples[idx]

    # Run full pipeline
    result = analyze_document(sample["policy_excerpt"], model, tokenizer)

    print(f"\n--- Sample {i+1}: {sample['id'][:50]} ---")
    print(f"Language:   {sample['language']}")
    print(f"True compliance:  {sample['overall_compliance']}")
    print(f"Pred compliance:  {result['overall_compliance']} (score: {result['overall_score']})")
    print(f"True gaps ({sample['gap_count']}): {[g for g, v in sample['gap_labels'].items() if v == 1]}")
    print(f"Pred gaps ({result['gap_count']}):")

    if result['password_policy']['gaps_detected']:
        print(f"  Password Policy ({result['password_policy']['gap_count']} gaps):")
        for d in result['password_policy']['details']:
            true_val = sample['gap_labels'].get(d['gap_id'], 0)
            marker = "✓" if true_val == 1 else "✗"
            print(f"    {marker} {d['gap_id']}: {d['description']} ({d['confidence']:.2%})")

    if result['risk_assessment']['gaps_detected']:
        print(f"  Risk Assessment ({result['risk_assessment']['gap_count']} gaps):")
        for d in result['risk_assessment']['details']:
            true_val = sample['gap_labels'].get(d['gap_id'], 0)
            marker = "✓" if true_val == 1 else "✗"
            print(f"    {marker} {d['gap_id']}: {d['description']} ({d['confidence']:.2%})")

    if result['gap_count'] == 0:
        print("  No gaps detected — fully compliant!")

# Show full JSON for last result
print(f"\n{'='*65}")
print("Full JSON output (last sample):")
print(json.dumps(result, indent=2, default=str))